# Qwen2.5-VL Tweet Media Enrichment
Replaces LLaVA (bakLlava-v1-hf) with Qwen2.5-VL-3B-Instruct on Lightning AI

In [ ]:
# CELL 1 — Install dependencies
!pip install -q -U \
    pandas \
    transformers \
    accelerate \
    bitsandbytes \
    requests \
    Pillow \
    tqdm \
    qwen-vl-utils

In [ ]:
# CELL 2 — Imports
import os
import re
import gc
import tempfile
import torch
import pandas as pd
import requests
from io import BytesIO
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# CELL 3 — Configuration
CSV_PATH      = "train.csv"
OUTPUT_PATH   = "train_enriched.csv"
MODEL_ID      = "Qwen/Qwen2.5-VL-3B-Instruct"
SAVE_EVERY    = 10       # save checkpoint every N rows
MAX_NEW_TOKENS = 150
IMAGE_SIZE    = (448, 448)   # fixed size — prevents CUDA fragmentation
TIMEOUT       = 10

# temp file — Lightning AI is Linux so /tmp is fine
TMP_IMG = os.path.join(tempfile.gettempdir(), "qwen_current.jpg")

VLM_PROMPT = (
    "Extract all visible text from this image and describe "
    "the scene briefly for a marketing tweet."
)

print(f"Model    : {MODEL_ID}")
print(f"Input    : {CSV_PATH}")
print(f"Output   : {OUTPUT_PATH}")
print(f"Temp img : {TMP_IMG}")

In [ ]:
# CELL 4 — Load model
print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("✅ Processor loaded")

print("Loading Qwen2.5-VL with 4-bit quantization...")
quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=quant_cfg,
    low_cpu_mem_usage=True,
)
model.eval()
print("✅ Model loaded!")
print(f"   Device map: {model.hf_device_map}")

In [ ]:
# CELL 5 — Helper functions

def extract_image_url(media_string):
    """Pull best image URL from media cell."""
    if not isinstance(media_string, str):
        return None
    # Photo full quality
    m = re.search(r"fullUrl='(https?://[^']+)'", media_string)
    if m:
        return m.group(1)
    # Video thumbnail fallback
    m = re.search(r"thumbnailUrl='(https?://[^']+)'", media_string)
    if m:
        return m.group(1)
    # Bare URL fallback
    if media_string.startswith("http"):
        return media_string
    return None


def download_to_disk(url, save_path):
    """
    Download image from url → resize to IMAGE_SIZE → save to save_path.
    Fixed size means pixel_values is always the same shape → no CUDA fragmentation.
    Returns True on success.
    """
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, timeout=TIMEOUT, headers=headers, stream=True)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB").resize(IMAGE_SIZE)
        img.save(save_path, format="JPEG")
        return True
    except Exception as e:
        print(f"  [download failed] {e}")
        return False


def run_vlm(image_path, prompt):
    """
    Feed saved image file to Qwen2.5-VL, return description string.

    Flow:
      image_path (str) → messages dict
      → process_vision_info opens file → [PIL.Image]
      → processor encodes → pixel_values tensor (GPU)
      → model.generate() → token ids (GPU)
      → batch_decode() → plain string
      → del ALL tensors + empty_cache()   ← every single row
    """
    image_inputs  = None
    video_inputs  = None
    inputs        = None
    generated_ids = None
    result        = ""

    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},  # file path string
                    {"type": "text",  "text": prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Opens image_path from disk → returns [PIL.Image]
        image_inputs, video_inputs = process_vision_info(messages)

        # Encodes PIL → pixel_values tensor + tokenises text → input_ids
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        # Free CPU image objects — no longer needed after processor()
        del image_inputs, video_inputs
        image_inputs = video_inputs = None

        # Move to GPU
        device = next(model.parameters()).device
        inputs = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in inputs.items()
        }

        # Clamp input_ids to vocab — prevents device-side assert from bad token IDs
        vocab_size = model.config.vocab_size
        inputs["input_ids"] = inputs["input_ids"].clamp(0, vocab_size - 1)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

        trimmed = [
            out[len(inp):]
            for inp, out in zip(inputs["input_ids"], generated_ids)
        ]
        result = processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        del trimmed

    finally:
        # Free EVERYTHING from GPU after every single row
        if inputs        is not None: del inputs
        if generated_ids is not None: del generated_ids
        if image_inputs  is not None: del image_inputs
        if video_inputs  is not None: del video_inputs
        try: del trimmed
        except NameError: pass
        gc.collect()
        torch.cuda.empty_cache()

    return result


def delete_temp(path):
    try:
        if os.path.exists(path):
            os.remove(path)
    except Exception:
        pass


print("✅ Helper functions defined")

In [ ]:
# CELL 6 — Load data + resume logic
print(f"Loading {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)
print(f"✅ Loaded {len(df)} rows")
print(f"   Columns: {df.columns.tolist()}")

if 'vlm_description' not in df.columns:
    df['vlm_description'] = ""

# Resume: if output file exists, copy already-done descriptions back in
if os.path.exists(OUTPUT_PATH):
    saved = pd.read_csv(OUTPUT_PATH)
    if 'vlm_description' in saved.columns and len(saved) == len(df):
        df['vlm_description'] = saved['vlm_description'].values
        print(f"🔄 Resumed from existing {OUTPUT_PATH}")

needs   = df['vlm_description'].isna() | (df['vlm_description'] == '')
indices = df.index[needs].tolist()
print(f"\nAlready done : {len(df) - len(indices)}")
print(f"Remaining    : {len(indices)} / {len(df)}")

In [ ]:
# CELL 7 — Main processing loop
success_count  = 0
fail_count     = 0
no_media_count = 0
loop_n         = 0

pbar = tqdm(indices, desc="VLM Enrichment")

try:
    for idx in pbar:
        row = df.iloc[idx]
        url = extract_image_url(str(row.get('media', '')))

        print(f"\n{'─'*55}")
        print(f"  Row {idx} | @{row.get('username','?')} | loop #{loop_n+1}")

        # No URL
        if not url:
            print(f"  [skip] no media")
            df.at[idx, 'vlm_description'] = "no media"
            no_media_count += 1
            loop_n += 1
            continue

        print(f"  [url]  {url[:75]}")

        # STEP 1 — Download → resize → save to disk
        ok = download_to_disk(url, TMP_IMG)
        if not ok:
            df.at[idx, 'vlm_description'] = "media could not be processed"
            fail_count += 1
            loop_n += 1
            continue

        size_kb = os.path.getsize(TMP_IMG) / 1024
        print(f"  [saved] {TMP_IMG}  ({size_kb:.1f} KB)")

        # STEP 2 — Feed path to Qwen, get description
        try:
            desc = run_vlm(TMP_IMG, VLM_PROMPT)
            df.at[idx, 'vlm_description'] = desc if desc else "media could not be processed"
            print(f"  [output] {desc[:150]}")
            success_count += 1
        except Exception as e:
            print(f"  [vlm error] {e}")
            df.at[idx, 'vlm_description'] = "media could not be processed"
            fail_count += 1
        finally:
            # STEP 3 — Delete temp file immediately after output
            delete_temp(TMP_IMG)
            print(f"  [deleted] {TMP_IMG}")

        loop_n += 1

        # Checkpoint save every SAVE_EVERY rows
        if loop_n % SAVE_EVERY == 0:
            df.to_csv(OUTPUT_PATH, index=False)
            df.to_csv(CSV_PATH, index=False)     # also update train.csv in place
            tqdm.write(f"  💾 Checkpoint: {loop_n} rows done → {OUTPUT_PATH}")

except KeyboardInterrupt:
    print("\n⚠️ Interrupted — saving progress...")

finally:
    delete_temp(TMP_IMG)    # safety net
    df.to_csv(OUTPUT_PATH, index=False)
    df.to_csv(CSV_PATH, index=False)
    print(f"\n💾 Final save → {OUTPUT_PATH}")
    print(f"💾 Final save → {CSV_PATH}")

In [ ]:
# CELL 8 — Stats + preview
valid = df['vlm_description'].notna() & ~df['vlm_description'].isin(
    ['', 'no media', 'media could not be processed']
)

print("=" * 55)
print("  FINAL STATS")
print("=" * 55)
print(f"  Total rows   : {len(df)}")
print(f"  Valid desc   : {valid.sum()}")
print(f"  No media     : {(df['vlm_description'] == 'no media').sum()}")
print(f"  Failed       : {(df['vlm_description'] == 'media could not be processed').sum()}")
print(f"  Still pending: {df['vlm_description'].isna().sum() + (df['vlm_description'] == '').sum()}")
if (success_count + fail_count) > 0:
    print(f"  Success rate : {success_count / (success_count + fail_count) * 100:.1f}%")
print("=" * 55)

print("\nSample outputs:")
for _, r in df[valid].head(5).iterrows():
    print(f"  @{r['username']}: {r['vlm_description'][:100]}")

print("\nDataframe preview:")
display(df[['username', 'content', 'vlm_description']].head(3))

In [ ]:
# CELL 9 — Cleanup GPU (run before training)
del model, processor
gc.collect()
torch.cuda.empty_cache()
print("✅ GPU memory cleared — ready for fine-tuning!")